### Notebook to check the Licenses of all Zenodo Records in our Cache and upload a ReadMe to Huggingface with the Corresponding Licencse Terms

In [ ]:
import sys
import os

# Add the root directory to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [ ]:
import requests

def get_zenodo_license(record_id):
    url = f"https://zenodo.org/api/records/{record_id}"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        license_info = data.get("metadata", {}).get("license", {})
        
        if license_info:
            license_id = license_info.get("id", "No ID available")
            return license_id
        else:
            return "No license information found."
    else:
        return f"Error: Unable to fetch data (Status Code: {response.status_code})"


## Load the Data
Load the DataSet from Huggingface and convert it to a pandas data frame (automatically done by the function).

In [ ]:
from caching import load_full_hf_cache
import pandas as pd

repo_name = "ScaDS-AI/SlideInsight_Cache_v2"

df = load_full_hf_cache(repo_name=repo_name)

In [ ]:
df.head()

## Extract all unique IDs

In [ ]:
unique_zenodo_ids = df["zenodo_record_id"].unique()
print(unique_zenodo_ids)
print(f"Number of total records: {len(unique_zenodo_ids)}")

Create a Set to ensure that all entries have the same license

## Now create the ReadMe file with the following information:
    - Links to the original zenodo records
    - Authors
    - Licenses of the original records.

First, gather information for each record:

In [ ]:
records_info = []

for record in unique_zenodo_ids:
    
    url = f"https://zenodo.org/api/records/{record}"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        authors = ", ".join([creator.get("name", "Unknown Author") for creator in data.get("metadata", {}).get("creators", [])])           
        record_url = data.get("links", {}).get("html", f"https://zenodo.org/record/{record}")
    else:
        print(f"Error: Unable to fetch data (Status Code: {response.status_code})")
    license = get_zenodo_license(record)
    
    records_info.append(f"- **[Zenodo Record {record}]({record_url})**  \n  **Authors**: {authors}  \n  **License**: {license}\n")

Second, combine it with some more general information (in Markdown Style)

In [ ]:
readme_content = (
    "# About this Dataset\n\n"
    "This Dataset contains data from several Presentation Slides as part of the [NFDI4BIOIMAGE](https://nfdi4bioimage.github.io/training/readme.html) "
    "project [SlideInsight](https://github.com/NFDI4BIOIMAGE/SlideInsight) to gain insights into presentation slides through multimodal AI models.\n\n"
    "For each Slide the following information is available:\n\n"
    "- key: recordID_pdfNumber_slideNumber\n\n"
    "- zenodo_record_id: record ID\n\n"
    "- zenodo_filename: PDF Filename\n\n"
    "- page_number: PDF Slide Number\n\n"
    "- text_embedding: Text Embedding (using the mixedbread-ai/mxbai-embed-large-v1 model)\n\n"
    "- visual_embedding: Vision Embedding (using the openai/clip-vit-base-patch32 model)\n\n"
    "- mixed_embedding: Mixed Embedding (text embedding of a generated structured description of the slide in JSON Format, using Qwen3-VL-8B-Instruct model)\n\n"
    "- structured_description: JSON-formatted Description of the Slides used for the mixed Embedding\n\n"
    "- extracted_text: Raw Text that was extracted from the Slide\n\n"
    "\n"
    "The corresponding images can be found in this [Huggingface Dataset](https://huggingface.co/datasets/ScaDS-AI/Slide_Insight_Images_v2)."
    "\n\n"
    "# Zenodo Records Information\n\n"
    "This repository contains data from Zenodo records.\n\n"
    "## Records\n\n" +
    "\n".join(records_info)
)

Third, upload the combined information as a valid Markdown file to the Huggingface Repository

In [ ]:
with open("HUGGINGFACE_README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

#### this *HUGGINGFACE_README.md* can now be used as the ReadMe file in the Huggingface project!